In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import requests
import re
from deep_translator import GoogleTranslator
import os
import time
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import os
import time
from joblib import Parallel, delayed
import psutil

## NLP

In [ ]:
# model_name = "ProsusAI/finbert"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(model_name)

# def analyze_sentiment(text):
#     inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
#     with torch.no_grad():
#         probs = torch.nn.functional.softmax(model(**inputs).logits, dim=-1)[0]
#     label_map = ["Negative", "Neutral", "Positive"]
#     pred_id = torch.argmax(probs).item()
#     return [label_map[pred_id], probs[pred_id].item()]


In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

def analyze_sentiment(text):
    # Разбиваем текст на предложения (после точек, восклицательных или вопросительных знаков)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    chunks = []
    current_chunk = ""
    
    # Группируем предложения так, чтобы суммарное число токенов в чанке не превышало 512.
    for sentence in sentences:
        # Получаем число токенов в текущем чанке и в новом предложении
        current_tokens = len(tokenizer.encode(current_chunk, add_special_tokens=False))
        sentence_tokens = len(tokenizer.encode(sentence, add_special_tokens=False))
        # Если добавление нового предложения превысит лимит и в текущем чанке уже есть текст, фиксируем его
        if current_chunk and current_tokens + sentence_tokens > 512:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += " " + sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    # Для каждого чанка получаем логиты и суммируем их
    logits_sum = None
    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            logits = model(**inputs).logits[0]
        if logits_sum is None:
            logits_sum = logits
        else:
            logits_sum += logits

    # Применяем softmax к суммарным логитам
    probs = torch.nn.functional.softmax(logits_sum, dim=-1)
    label_map = ["Negative", "Neutral", "Positive"]
    pred_id = torch.argmax(probs).item()
    return [label_map[pred_id], probs[pred_id].item()]

## Translate

In [ ]:
def translate_russian_to_english(text):
    translator = GoogleTranslator(source='ru', target='en')
    return " ".join(filter(None, [translator.translate(text[i:i+1000]) for i in range(0, len(text), 1000)]))

In [ ]:
def translate_russian_to_english(text):
    import time
    translator = GoogleTranslator(source='ru', target='en')
    result = []
    chunk_size = 1000
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]
        success = False
        while not success:
            try:
                translated = translator.translate(chunk)
                result.append(translated)
                success = True  # Exit loop when successful
            except Exception as e:
                error_str = str(e).lower()
                if "timeout" in error_str:
                    print(f"Timeout error occurred on chunk starting at {i}: {e}. Waiting 60 seconds to retry...")
                    time.sleep(10)
                else:
                    print(f"Error occurred on chunk starting at {i}: {e}. Waiting 10 seconds to retry...")
                    time.sleep(1)
    return " ".join(filter(None, result))

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

def translate_offline(text, model_name="Helsinki-NLP/opus-mt-ru-en"):
    tokenizer, model = MarianTokenizer.from_pretrained(model_name), MarianMTModel.from_pretrained(model_name)
    sentences = text.split(". ")
    chunks, chunk, translated_texts = [], "", []

    for sentence in sentences:
        chunk += sentence + ". "
        if len(chunk) > 512:
            chunks.append(chunk.strip())
            chunk = ""
    if chunk: chunks.append(chunk.strip())

    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=512)
        translated_texts.append(tokenizer.decode(model.generate(**inputs)[0], skip_special_tokens=True))

    return " ".join(translated_texts)

## News processing Interfax

In [ ]:
import pandas as pd
import os
import time
from joblib import Parallel, delayed
import psutil
memory_limit = int(psutil.virtual_memory().total * 0.8 / (16 * 1024**2))  # 50% памяти на 4 процесса


folder = r"../../Data/Processed_data/Interfax_news_database/Interfax_downloaded_news"
merged_file = 'merged_data.csv'
output_folder = r"../../Data/Processed_data/Interfax_news_database/Interfax_downloaded_news_processed"

data = pd.read_csv(os.path.join(folder, merged_file), encoding='UTF-16', delimiter='^')
data = data.dropna(subset='news_content')

def process_row(i, row):
    """Перевод и анализ одной строки с обработкой ошибок."""
    retries = 3
    while retries > 0:
        try:
            text = translate_russian_to_english(row['news_content'])
            return i, text, *analyze_sentiment(text)
        except ConnectionResetError as e:
            print(f"Ошибка сети {i}: {e}. Жду 30 секунд перед повтором...")
            time.sleep(30)
            retries -= 1
        except Exception as e:
            print(f"Ошибка {i}: {e}. Повтор через 10 сек")
            time.sleep(10)
    
    return i, None, None, None  # Если все попытки провалились

chunk_size = 100
chunks = [data.iloc[i:i+chunk_size].copy() for i in range(0, len(data), chunk_size)]

n = 600
for idx, chunk in enumerate(chunks[n:700], n):
    print(f"обработка {idx}/{len(chunks)}...")

    for col in ["translation", "sentiment", "prob"]:
        if col not in chunk.columns:
            chunk[col] = None  # Создаем столбцы, если их нет

    results = Parallel(n_jobs=8, max_nbytes=f"{memory_limit}M", backend="loky")(delayed(process_row)(i, row) for i, row in chunk.iterrows())

    for i, text, sentiment, prob in results:
        chunk.at[i, "translation"] = text
        chunk.at[i, "sentiment"] = sentiment
        chunk.at[i, "prob"] = prob

    chunk.to_csv(os.path.join(output_folder, f"processed_part_{idx}.csv"), encoding='utf-16', index=False, sep='^')

## News processing 1prime

In [ ]:
import pandas as pd
import os
import time
from joblib import Parallel, delayed
import psutil
memory_limit = int(psutil.virtual_memory().total * 0.8 / (16 * 1024**2))


folder = r"../../Data/Processed_data/1prime_news_database/Narrowed_news_sample_v2"
merged_file = 'merged_data.csv'
output_folder = r"../../Data/Processed_data/1prime_news_database/1prime_downloaded_news_processed"

data = pd.read_csv(os.path.join(folder, merged_file), encoding='UTF-16', delimiter='^')
data = data.dropna(subset='news_content')

def process_row(i, row):
    """Перевод и анализ одной строки с обработкой ошибок."""
    retries = 3
    while retries > 0:
        try:
            text = translate_russian_to_english(row['news_content'])
            return i, text, *analyze_sentiment(text)
        except ConnectionResetError as e:
            print(f"Ошибка сети {i}: {e}. Жду 30 секунд перед повтором...")
            time.sleep(30)
            retries -= 1
        except Exception as e:
            print(f"Ошибка {i}: {e}. Повтор через 10 сек")
            time.sleep(10)
    
    return i, None, None, None  # Если все попытки провалились

chunk_size = 100
chunks = [data.iloc[i:i+chunk_size].copy() for i in range(0, len(data), chunk_size)]

n = 0
for idx, chunk in enumerate(chunks[n:], n):
    print(f"обработка {idx}/{len(chunks)}...")

    for col in ["translation", "sentiment", "prob"]:
        if col not in chunk.columns:
            chunk[col] = None  # Создаем столбцы, если их нет

    results = Parallel(n_jobs=8, max_nbytes=f"{memory_limit}M", backend="loky")(delayed(process_row)(i, row) for i, row in chunk.iterrows())

    for i, text, sentiment, prob in results:
        chunk.at[i, "translation"] = text
        chunk.at[i, "sentiment"] = sentiment
        chunk.at[i, "prob"] = prob

    chunk.to_csv(os.path.join(output_folder, f"processed_part_{idx}.csv"), encoding='utf-16', index=False, sep='^')

## News processing RBC

In [ ]:
import pandas as pd
import os
import time
from joblib import Parallel, delayed
import psutil
memory_limit = int(psutil.virtual_memory().total * 0.8 / (16 * 1024**2))


folder = r"../../Data/Processed_data/RBC_news_database/RBC_downloaded_news"
merged_file = 'merged_data.csv'
output_folder = r"../../Data/Processed_data/RBC_news_database/RBC_downloaded_news_processed"

data = pd.read_csv(os.path.join(folder, merged_file), encoding='UTF-16', delimiter='^')
data = data.dropna(subset='news_content')

def process_row(i, row):
    """Перевод и анализ одной строки с обработкой ошибок."""
    retries = 3
    while retries > 0:
        try:
            text = translate_russian_to_english(row['news_content'])
            return i, text, *analyze_sentiment(text)
        except ConnectionResetError as e:
            print(f"Ошибка сети {i}: {e}. Жду 30 секунд перед повтором...")
            time.sleep(30)
            retries -= 1
        except Exception as e:
            print(f"Ошибка {i}: {e}. Повтор через 10 сек")
            time.sleep(10)
    
    return i, None, None, None  # Если все попытки провалились

chunk_size = 100
chunks = [data.iloc[i:i+chunk_size].copy() for i in range(0, len(data), chunk_size)]

n = 57
for idx, chunk in enumerate(chunks[n:], n):
    print(f"обработка {idx}/{len(chunks)}...")

    for col in ["translation", "sentiment", "prob"]:
        if col not in chunk.columns:
            chunk[col] = None  # Создаем столбцы, если их нет

    results = Parallel(n_jobs=8, max_nbytes=f"{memory_limit}M", backend="loky")(delayed(process_row)(i, row) for i, row in chunk.iterrows())

    for i, text, sentiment, prob in results:
        chunk.at[i, "translation"] = text
        chunk.at[i, "sentiment"] = sentiment
        chunk.at[i, "prob"] = prob

    chunk.to_csv(os.path.join(output_folder, f"processed_part_{idx}.csv"), encoding='utf-16', index=False, sep='^')

In [ ]:
def prime_url_processing(url):
    headers = {"User-Agent": "Mozilla/5.0"}  # Helps avoid blocks

    response = requests.get(url, headers=headers)
    page_source = response.text
    
    bold_texts = re.findall(r'itemprop="articleBody">(.*?)</div>', page_source, re.DOTALL)[0]
    words = [word for text in bold_texts for word in text.split()]
    
    return bold_texts

## Finbert on missed data + GPU

In [ ]:
data = pd.read_excel(r'../../Data/Processed_data/missed_news/missed_news_all.xlsx')

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm

# Parallelize the translation over the news content rows
translations = Parallel(n_jobs=4)(
    delayed(translate_russian_to_english)(content) 
    for content in tqdm(data['news_content'], desc="Translating")
)

data['translation'] = translations

In [ ]:
# data.dropna(subset='news_content').to_excel(r'../../Data/Processed_data/missed_news/missed_news_all_translated.xlsx',
# index=False,
#     engine='xlsxwriter',
#     engine_kwargs={'options': {'strings_to_urls': False}}
# )

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

def analyze_sentiment(text, threshold=0.8):
    # 1) Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text)

    # 2) Build chunks ≤512 tokens
    chunks, curr = [], ""
    for sent in sentences:
        ntok = len(tokenizer.encode(curr, add_special_tokens=False))
        stok = len(tokenizer.encode(sent, add_special_tokens=False))
        if curr and ntok + stok > 512:
            chunks.append(curr.strip())
            curr = sent
        else:
            curr += " " + sent
    if curr:
        chunks.append(curr.strip())

    # 3) Sum logits over all chunks
    logits_sum = None
    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            logits = model(**inputs).logits[0]
        logits_sum = logits if logits_sum is None else logits_sum + logits

    # 4) Compute softmax probabilities
    probs = torch.nn.functional.softmax(logits_sum, dim=-1)
    neg, neu, pos = probs.tolist()

    # 5) Apply thresholding
    if pos > threshold:
        label, score = "Positive", pos
    elif neg > threshold*0.5:
        label, score = "Negative", neg
    else:
        label, score = "Neutral", neu

    return label, {"negative": neg, "neutral": neu, "positive": pos}


In [ ]:
import re
from collections import Counter
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

def analyze_sentiment(text):
    # 1) Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text)

    # 2) Build chunks ≤512 tokens
    chunks, curr = [], ""
    for sent in sentences:
        ntok = len(tokenizer.encode(curr, add_special_tokens=False))
        stok = len(tokenizer.encode(sent, add_special_tokens=False))
        if curr and ntok + stok > 512:
            chunks.append(curr.strip())
            curr = sent
        else:
            curr += " " + sent
    if curr:
        chunks.append(curr.strip())

    # 3) Classify each chunk independently
    label_map = ["Negative", "Neutral", "Positive"]
    
    chunk_labels = []

    threshold = 0.9
    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            logits = model(**inputs).logits[0]
        neg, neu, pos = torch.nn.functional.softmax(logits, dim=-1).tolist()
        
        if pos  > threshold:
            pred = "Positive"
        elif neg > threshold*0.5:
            pred = "Negative"
        else:
            pred = "Neutral"
        
        chunk_labels.append(pred)

    # 4) Final sentiment is the most frequent chunk label
    final_label = Counter(chunk_labels).most_common(1)[0][0]
    return final_label, dict(Counter(chunk_labels))

In [ ]:
# data = pd.read_excel(r'../../Data/Processed_data/missed_news/missed_news_all_translated.xlsx')

In [ ]:
data = pd.read_excel(r'../../Data/All_news_FBRT_processed.xlsx')

In [ ]:
# data = data.head(100)

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm

# Создаем столбцы, если их нет
# for col in ["sentiment", "prob"]:
#     if col not in data.columns:
#         data[col] = None
data[["sentiment", "prob"]] = None, None 

# Локальная обертка для безопасного вызова analyze_sentiment без изменения функции
def safe_analyze_sentiment(text):
    try:
        return analyze_sentiment(text)
    except:
        return ("Neutral", 0.0)

results = Parallel(n_jobs=1, backend="loky")(
    delayed(safe_analyze_sentiment)(row['translation']) 
    for idx, row in tqdm(data.iterrows(), total=len(data), desc="Analyzing sentiment")
)

for idx, (sentiment, prob) in enumerate(results):
    data.at[idx, "sentiment"] = sentiment
    data.at[idx, "prob"] = prob

In [ ]:
# data.to_excel(r'../../Data/Processed_data/missed_news/missed_news_all_FNBRT.xlsx', index=False, engine='xlsxwriter', engine_kwargs={'options': {'strings_to_urls': False}})

In [ ]:
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)


inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
with torch.no_grad():
    probs = torch.nn.functional.softmax(model(**inputs).logits, dim=-1)[0]
label_map = ["Negative", "Neutral", "Positive"]
pred_id = torch.argmax(probs).item()
print(label_map[pred_id], probs[pred_id].item())

In [ ]:
text = 'ВТБ сообщил о росте выручки. Роснефть объявила о сокращении расходов. VTB announced higher revenue; Rosneft reported lower costs.'

In [ ]:
a = inputs['input_ids']

In [ ]:
# from tqdm import tqdm
# import torch
# from transformers import AutoTokenizer, AutoModelForSequenceClassification

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model_name = "ProsusAI/finbert"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

# def process_text(text):
#     sentences = text.split(". ")
#     chunks, current = [], ""
#     max_len = 510
#     for sent in sentences:
#         cand = f"{current}{'. ' if current else ''}{sent}"
#         if len(tokenizer.encode(cand, add_special_tokens=True)) <= max_len:
#             current = cand
#         else:
#             if current:
#                 chunks.append(current.strip())
#             tokens = tokenizer.encode(sent, add_special_tokens=True)
#             if len(tokens) > max_len:
#                 tokens = tokens[:max_len]
#                 sent = tokenizer.decode(tokens, skip_special_tokens=True)
#             current = sent
#     if current:
#         chunks.append(current.strip())
    
#     total_logits = None
#     for ch in chunks:
#         inputs = tokenizer(ch, return_tensors="pt", truncation=True, max_length=512)
#         inputs = {k: v.to(device) for k, v in inputs.items()}
#         with torch.no_grad():
#             logits = model(**inputs).logits[0]
#         total_logits = logits if total_logits is None else total_logits + logits
#     probs = torch.nn.functional.softmax(total_logits, dim=-1)
#     labels = ["Negative", "Neutral", "Positive"]
#     idx = torch.argmax(probs).item()
#     return labels[idx], probs[idx].item()

In [ ]:
# output_folder = r'../../Data/Processed_data/missed_news/processed'

# def analyze_sentiment(i, row):
#     """Перевод и анализ одной строки с обработкой ошибок."""
#     retries = 3
#     while retries > 0:
#         try:
#             text = translate_russian_to_english(row['news_content'])
#             return i, text, *process_text(text)
#         except ConnectionResetError as e:
#             print(f"Ошибка сети {i}: {e}. Жду 30 секунд перед повтором...")
#             time.sleep(30)
#             retries -= 1
#         except Exception as e:
#             print(f"Ошибка {i}: {e}. Повтор через 10 сек")
#             time.sleep(10)
#     print(1)
#     return i, None, None, None  # Если все попытки провалились

# chunk_size = 10
# chunks = [data.iloc[i:i+chunk_size].copy() for i in range(0, len(data), chunk_size)]

# n = 0

# from tqdm import tqdm

# for idx, chunk in enumerate(chunks[n:], n):
#     print(f"обработка {idx}/{len(chunks)}...")

#     for col in ["translation", "sentiment", "prob"]:
#         if col not in chunk.columns:
#             chunk[col] = None  # Создаем столбцы, если их нет

#     results = Parallel(n_jobs=8, backend="loky")(
#         delayed(analyze_sentiment)(i, row)
#         for i, row in tqdm(chunk.iterrows(), total=len(chunk), desc="Processing rows", leave=False)
#     )

#     for i, text, sentiment, prob in results:
#         chunk.at[i, "translation"] = text
#         chunk.at[i, "sentiment"] = sentiment
#         chunk.at[i, "prob"] = prob

#     chunk.to_excel(os.path.join(output_folder, f"processed_part_{idx}.xlsx"), index=False)

## SandBox

### 1Prime

In [ ]:
url = 'https://1prime.ru/20250109/vtb--854100814.html'
etxt = translate_russian_to_english(prime_url_processing(url)).replace("'",'').replace('"','')
analyze_sentiment(etxt)

In [ ]:
folder = r"../../Data/Processed_data/RBC_news_database/RBC_downloaded_news"
csv_files = list(filter(lambda f: f.endswith(".csv"), os.listdir(folder)))

In [ ]:
import pandas as pd
import os

folder = r"../../Data/Processed_data/RBC_news_database/RBC_downloaded_news"
csv_files = [f for f in os.listdir(folder) if f.endswith(".csv")]

df_list = []
for file in csv_files:
    df = pd.read_csv(os.path.join(folder, file), encoding="UTF-16", delimiter="^")
    df.insert(0, "source_file", file[:-4])  # Добавляем колонку с названием файла (без .csv)
    df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True).drop_duplicates(subset="link").dropna(subset='news_content')
combined_df = combined_df[~combined_df['news_content'].isin(['no_data','external_source'])]
# combined_df['source_file'] = combined_df['source_file'].apply(lambda x: int(x))
combined_df = combined_df.sort_values(by='source_file', ascending=True)
# Сохранение в CSV
output_path = os.path.join(folder, "merged_data.csv")
combined_df.to_csv(output_path, encoding="UTF-16", index=False, sep="^")

print(f"Объединено файлов: {len(csv_files)}, итоговых строк: {len(combined_df)}")


In [ ]:
for file in csv_files[3:4]:
    data = pd.read_csv(os.path.join(folder, file), encoding='UTF-16', delimiter='^')
    data['sentiment'] = None
    data['prob'] = None
    
    for i in range(len(data)):
        processed = False
        while not processed:
            try:
                translated_text = translate_russian_to_english(data['news_content'][i])
                data.loc[i, 'sentiment'], data.loc[i, 'prob'] = analyze_sentiment(translated_text)
                print(i)
                processed = True
            except Exception as e:
                print(f"Error processing row {i}: {e}. Retrying...")
                time.sleep(10)
    
    output_path = fr'../../Data/Processed_data/Interfax_news_database/Interfax_downloaded_news_processed/{file}'
    data.to_csv(output_path, encoding='utf-16', index=False, sep='^')